In [1]:
import requests
import pandas as pd
import time
from dotenv import load_dotenv
import os

In [2]:
# Load API key from .env file
load_dotenv()
API_KEY = os.getenv("YOUTUBE_API_KEY")

# Search keyword
QUERY = "Artificial Intelligence"

# Maximum results per API call (YouTube limit is 50)
MAX_RESULTS = 50

# Target number of videos to collect
TARGET_COUNT = 120

# List to store collected video data
videos_data = []

In [9]:
# Function: Search for videos using search.list
def search_videos(query, page_token=None):
    url = "https://www.googleapis.com/youtube/v3/search"
    params = {
        "part": "snippet",
        "q": query,
        "type": "video",
        "maxResults": MAX_RESULTS,
        "order": "date",
        "key": API_KEY,
        "pageToken": page_token
    }
    response = requests.get(url, params=params)
    return response.json()

In [10]:
# Function: Get video details using videos.list
def get_video_details(video_ids):
    url = "https://www.googleapis.com/youtube/v3/videos"
    params = {
        "part": "snippet,statistics",
        "id": ",".join(video_ids),
        "key": API_KEY
    }
    response = requests.get(url, params=params)
    return response.json()

In [11]:
# Main loop: Continue collecting until target count is reached
next_page_token = None

while len(videos_data) < TARGET_COUNT:
    search_response = search_videos(QUERY, next_page_token)

    # Extract video IDs from search results
    video_ids = [item["id"]["videoId"] for item in search_response.get("items", [])]

    # Fetch detailed information for each video
    details_response = get_video_details(video_ids)

    for item in details_response.get("items", []):
        snippet = item["snippet"]
        stats = item.get("statistics", {})

        videos_data.append({
            "video_id": item["id"],
            "title": snippet.get("title"),
            "description": snippet.get("description"),
            "published_at": snippet.get("publishedAt"),
            "channel_title": snippet.get("channelTitle"),
            "view_count": stats.get("viewCount"),
            "like_count": stats.get("likeCount"),
            "comment_count": stats.get("commentCount"),
            "url": f"https://www.youtube.com/watch?v={item['id']}",
            "platform": "YouTube",  # Added platform field
            "keyword": QUERY
        })
    # Move to the next page of results
    next_page_token = search_response.get("nextPageToken")

    # Stop if no more pages are available
    if not next_page_token:
        break
    
    # Prevent sending requests too quickly
    time.sleep(1)

In [12]:
# Save collected data to CSV
df = pd.DataFrame(videos_data)
df.to_csv("youtube_videos.csv", index=False)

print(f"Done! Collected {len(videos_data)} videos.")

Done! Collected 144 videos.
